In [1]:
import os
import sys

# --- Fix: TensorFlow/protobuf import crash in Kaggle Py3.12
# Force pure-Python protobuf implementation before *any* protobuf/TF import occurs.
# This addresses: AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION"] = "2"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_DISABLE_UPB"] = "1"
os.environ["TF_PROTOBUF_DISABLE_CPP"] = "1"

# Keep existing runtime flags
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("PYTHONHASHSEED", "0")
os.environ.setdefault("TF_USE_CXX11_ABI", "0")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
os.environ.setdefault("TF_USE_LEGACY_KERAS", "0")

# Sanitize any pre-imported protobuf modules (can happen in some notebook environments)
for m in list(sys.modules.keys()):
    if m.startswith("google.protobuf"):
        sys.modules.pop(m, None)

try:
    import google.protobuf  # noqa: F401
except Exception as _e:
    print("Warning: could not import google.protobuf early:", repr(_e))

import numpy as np
import pandas as pd

np.random.seed(0)

PLATFORM = "kaggle"  # local kaggle
NEEDTRAIN = False
LOAD_MODELS_FROM = "models20240117"
if PLATFORM == "local":
    LOAD_MODELS_FROM = f"./input/{LOAD_MODELS_FROM}"
elif PLATFORM == "kaggle":
    LOAD_MODELS_FROM = f"/kaggle/input/{LOAD_MODELS_FROM}"

EEG_LENGTH = 20.48  # s
SFREQ = 200

HIGH = 128
LENGTH = 256
BRAIN = {
    "LL": ["Fp1-F7", "F7-T3", "T3-T5", "T5-O1"],
    "RL": ["Fp2-F8", "F8-T4", "T4-T6", "T6-O2"],
    "LP": ["Fp1-F3", "F3-C3", "C3-P3", "P3-O1"],
    "RP": ["Fp2-F4", "F4-C4", "C4-P4", "P4-O2"],
}


def _try_import_tensorflow():
    """
    Fix: isolate TF import so we can safely fall back without crashing the whole run.
    Keep core model logic unchanged when TF imports successfully.
    """
    try:
        # Re-assert env vars inside the helper (some environments mutate them)
        os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
        os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION"] = "2"
        os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_DISABLE_UPB"] = "1"
        os.environ["TF_PROTOBUF_DISABLE_CPP"] = "1"

        # Ensure no stale protobuf C++ bindings are left in sys.modules
        for m in list(sys.modules.keys()):
            if m.startswith("google.protobuf"):
                sys.modules.pop(m, None)

        import google.protobuf  # noqa: F401
        import tensorflow as tf  # noqa: F401

        return tf, None
    except Exception as e:
        return None, e


TF_AVAILABLE = True
tf, tf_err = _try_import_tensorflow()
if tf is None:
    TF_AVAILABLE = False
    strategy = None
    VER = 1
    print(
        "TensorFlow import failed; will use prior fallback only. Reason:", repr(tf_err)
    )
else:
    try:
        try:
            tf.keras.utils.set_random_seed(0)
        except Exception:
            pass

        print("TensorFlow version =", tf.__version__)

        gpus = tf.config.list_physical_devices("GPU")
        if len(gpus) == 0:
            strategy = tf.distribute.OneDeviceStrategy(device="/cpu:0")
            print("Using CPU")
        elif len(gpus) == 1:
            strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
            print(f"Using {len(gpus)} GPU")
        else:
            strategy = tf.distribute.MirroredStrategy()
            print(f"Using {len(gpus)} GPUs")

        VER = 1

        MIX = True
        if MIX:
            # Keep existing intent; make API compatible with current TF
            try:
                from tensorflow.keras import mixed_precision

                mixed_precision.set_global_policy("mixed_float16")
                print("Mixed precision enabled via set_global_policy")
            except Exception as e:
                try:
                    tf.config.optimizer.set_experimental_options(
                        {"auto_mixed_precision": True}
                    )
                    print("Mixed precision enabled via experimental optimizer option")
                except Exception as e2:
                    print(
                        "Mixed precision requested but could not be enabled:",
                        repr(e),
                        repr(e2),
                    )
        else:
            print("Using full precision")

    except Exception as e:
        TF_AVAILABLE = False
        tf = None
        strategy = None
        VER = 1
        print("TensorFlow setup failed; will use prior fallback only. Reason:", repr(e))



2026-05-15 11:58:13.503449: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778846293.516690      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778846293.520741      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TensorFlow version = 2.18.0
Using 1 GPU
Mixed precision enabled via set_global_policy


I0000 00:00:1778846300.501606      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:c5:00.0, compute capability: 8.6


In [2]:
if PLATFORM == "local":
    df = pd.read_csv("./input/hms-harmful-brain-activity-classification/train.csv")
elif PLATFORM == "kaggle":
    df = pd.read_csv(
        "/kaggle/input/hms-harmful-brain-activity-classification/train.csv"
    )

TARGETS = df.columns[-6:]
print("Train shape:", df.shape)
print("Targets", list(TARGETS))

train = df.groupby("eeg_id")[
    ["spectrogram_id", "spectrogram_label_offset_seconds", "eeg_label_offset_seconds"]
].agg(
    {
        "spectrogram_id": "first",
        "spectrogram_label_offset_seconds": "min",
        "eeg_label_offset_seconds": "median",
    }
)
train.columns = ["spec_id", "min", "eeg_median"]

tmp = df.groupby("eeg_id")[["spectrogram_id", "spectrogram_label_offset_seconds"]].agg(
    {"spectrogram_label_offset_seconds": "max"}
)
train["max"] = tmp

tmp = df.groupby("eeg_id")[["patient_id"]].agg("first")
train["patient_id"] = tmp

tmp = df.groupby("eeg_id")[TARGETS].agg("sum")
for t in TARGETS:
    train[t] = tmp[t].values

y_data = train[TARGETS].values
y_data = y_data / y_data.sum(axis=1, keepdims=True)
train[TARGETS] = y_data

tmp = df.groupby("eeg_id")[["expert_consensus"]].agg("first")
train["target"] = tmp

train = train.reset_index()
print("Train non-overlapp eeg_id shape:", train.shape)

for c in ["min", "max"]:
    train[c] = pd.to_numeric(train[c], errors="coerce").fillna(0).astype(int)

_global_prior = train[list(TARGETS)].sum().values.astype("float64")
_global_prior = _global_prior / _global_prior.sum()
print("Global label prior (per-eeg_id):", dict(zip(TARGETS, _global_prior.round(6))))

_patient_counts = train.groupby("patient_id")[list(TARGETS)].sum().astype("float64")
alpha = 0.3
_patient_prior = (
    (_patient_counts + alpha).div((_patient_counts + alpha).sum(axis=1), axis=0)
).fillna(0.0)
print("Patient priors computed for patients:", _patient_prior.shape[0])



Train shape: (96950, 15)
Targets ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
Train non-overlapp eeg_id shape: (15396, 13)
Global label prior (per-eeg_id): {'seizure_vote': 0.153466, 'lpd_vote': 0.140462, 'gpd_vote': 0.1045, 'lrda_vote': 0.066646, 'grda_vote': 0.116353, 'other_vote': 0.418572}
Patient priors computed for patients: 1910


In [3]:
try:
    import albumentations as albu
except Exception as e:
    albu = None
    print("albumentations not available; augmentation disabled. Reason:", repr(e))

if TF_AVAILABLE:
    TARS = {"Seizure": 0, "LPD": 1, "GPD": 2, "LRDA": 3, "GRDA": 4, "Other": 5}
    TARS2 = {x: y for y, x in TARS.items()}

    class DataGenerator(tf.keras.utils.Sequence):
        "Generates data for Keras"

        def __init__(
            self,
            data,
            batch_size=32,
            shuffle=False,
            augment=False,
            mode="train",
            specs=None,
            eegs=None,
        ):
            self.data = data
            self.batch_size = batch_size
            self.shuffle = shuffle
            self.augment = augment and (albu is not None)
            self.mode = mode
            self.specs = specs if specs is not None else {}
            self.eegs = eegs if eegs is not None else {}
            self.on_epoch_end()

        def __len__(self):
            ct = int(np.ceil(len(self.data) / self.batch_size))
            return ct

        def __getitem__(self, index):
            indexes = self.indexes[
                index * self.batch_size : (index + 1) * self.batch_size
            ]
            X, X_eeg, y = self.__data_generation(indexes)
            if self.augment:
                X = self.__augment_batch(X)
            return [X, X_eeg], y

        def on_epoch_end(self):
            self.indexes = np.arange(len(self.data))
            if self.shuffle:
                np.random.shuffle(self.indexes)

        def __data_generation(self, indexes):
            X = np.zeros((len(indexes), HIGH, LENGTH, 4), dtype="float32")
            X_eeg = np.zeros(
                (len(indexes), 4, round(EEG_LENGTH * SFREQ), 4), dtype="float32"
            )
            y = np.zeros((len(indexes), 6), dtype="float32")

            for j, i in enumerate(indexes):
                row = self.data.iloc[i]
                if self.mode == "test":
                    r = 0
                elif self.mode == "valid":
                    r = int((int(row["min"]) + int(row["max"])) // 4)
                else:
                    r = np.random.randint(int(row["min"]), int(row["max"]) + 1) // 2

                spec_arr = self.specs.get(int(row.spec_id), None)
                eeg_arr = self.eegs.get(int(row.eeg_id), None)

                for k in range(4):
                    if spec_arr is not None:
                        max_r = max(0, spec_arr.shape[0] - 300)
                        rr = int(np.clip(r, 0, max_r))
                        img = spec_arr[rr : rr + 300, k * 100 : (k + 1) * 100].T
                    else:
                        img = np.ones((100, 300), dtype="float32")

                    if eeg_arr is not None:
                        img_eeg = eeg_arr[:, :, k]
                    else:
                        img_eeg = np.ones(
                            (4, round(EEG_LENGTH * SFREQ)), dtype="float32"
                        )

                    img = np.clip(img, np.exp(-4), np.exp(8))
                    img = np.log(img)
                    img = np.nan_to_num(img, nan=0.0)

                    if HIGH <= 100:
                        img = np.resize(
                            img[
                                :,
                                round((600 / 2 - LENGTH) / 2) : (
                                    round((600 / 2 - LENGTH) / 2) + LENGTH
                                ),
                            ],
                            (HIGH, LENGTH),
                        )
                    else:
                        img = img[
                            :,
                            round((600 / 2 - LENGTH) / 2) : (
                                round((600 / 2 - LENGTH) / 2) + LENGTH
                            ),
                        ]

                    ep = 1e-6
                    m = np.nanmean(img.flatten())
                    s = np.nanstd(img.flatten())
                    img = (img - m) / (s + ep)
                    img = np.nan_to_num(img, nan=0.0)

                    if HIGH <= 100:
                        X[j, :, :, k] = img
                    else:
                        X[
                            j, round((HIGH - 100) / 2) : -round((HIGH - 100) / 2), :, k
                        ] = img

                    X_eeg[j, :, :, k] = img_eeg

                if self.mode != "test":
                    y[j] = row[TARGETS].values

            return X, X_eeg, y

        def __random_transform(self, img):
            composition = albu.Compose(
                [
                    albu.HorizontalFlip(p=0.5),
                    albu.CoarseDropout(
                        max_holes=8, max_height=32, max_width=32, fill_value=0, p=0.5
                    ),
                ]
            )
            return composition(image=img)["image"]

        def __augment_batch(self, img_batch):
            for i in range(img_batch.shape[0]):
                img_batch[i,] = self.__random_transform(img_batch[i,])
            return img_batch




In [4]:
if TF_AVAILABLE:
    from tensorflow.keras import layers

    def build_model():
        inp = tf.keras.Input(shape=(HIGH, LENGTH, 4))
        inp_eeg = tf.keras.Input(shape=(4, round(EEG_LENGTH * SFREQ), 4))

        base_model = tf.keras.applications.EfficientNetB2(
            include_top=False, weights="imagenet"
        )
        base_model_eeg = tf.keras.applications.EfficientNetB1(
            include_top=False, weights="imagenet"
        )

        x0 = inp[:, :, :, :1]
        x1 = inp[:, :, :, 1:2]
        x2 = inp[:, :, :, 2:3]
        x3 = inp[:, :, :, 3:4]
        x = layers.Concatenate(axis=1)([x0, x1, x2, x3])
        x = layers.Concatenate(axis=3)([x, x, x])

        x0_eeg = inp_eeg[:, :, :, :1]
        x1_eeg = inp_eeg[:, :, :, 1:2]
        x2_eeg = inp_eeg[:, :, :, 2:3]
        x3_eeg = inp_eeg[:, :, :, 3:4]
        x_eeg = layers.Concatenate(axis=1)([x0_eeg, x1_eeg, x2_eeg, x3_eeg])
        x_eeg = layers.Reshape((-1, 256, 1))(x_eeg)
        x_eeg = layers.Concatenate(axis=3)([x_eeg, x_eeg, x_eeg])

        x = base_model(x)
        x = layers.GlobalAveragePooling2D()(x)

        x_eeg = base_model_eeg(x_eeg)
        x_eeg = layers.GlobalAveragePooling2D()(x_eeg)

        x = layers.Concatenate(axis=1)([x, x_eeg])
        x = layers.Dense(6, activation="softmax", dtype="float32")(x)

        model = tf.keras.Model(inputs=[inp, inp_eeg], outputs=x)
        opt = tf.keras.optimizers.Adam(learning_rate=1e-3)
        loss = tf.keras.losses.CategoricalCrossentropy()
        model.compile(loss=loss, optimizer=opt)
        return model




In [5]:
if PLATFORM == "local":
    test = pd.read_csv("./input/hms-harmful-brain-activity-classification/test.csv")
elif PLATFORM == "kaggle":
    test = pd.read_csv(
        "/kaggle/input/hms-harmful-brain-activity-classification/test.csv"
    )
print("Test shape", test.shape)

if PLATFORM == "local":
    PATH_SPEC = "./input/hms-harmful-brain-activity-classification/test_spectrograms/"
elif PLATFORM == "kaggle":
    PATH_SPEC = (
        "/kaggle/input/hms-harmful-brain-activity-classification/test_spectrograms/"
    )

test = test.rename({"spectrogram_id": "spec_id"}, axis=1)
needed_spec_ids = set(test.spec_id.astype(int).tolist())

files_spec = os.listdir(PATH_SPEC)
print(f"There are {len(files_spec)} test spectrogram parquets")

spectrograms2 = {}
loaded = 0
for i, f in enumerate(files_spec):
    if i % 200 == 0:
        print(i, ", ", end="")
    try:
        name = int(f.split(".")[0])
    except Exception:
        continue
    if name not in needed_spec_ids:
        continue
    tmp = pd.read_parquet(f"{PATH_SPEC}{f}")
    spectrograms2[name] = tmp.iloc[:, 1:].values
    loaded += 1

print(f"\nLoaded spectrograms: {loaded}/{len(needed_spec_ids)} needed")



Test shape (9850, 3)
There are 1114 test spectrogram parquets
0 , 200 , 400 , 600 , 800 , 1000 , 
Loaded spectrograms: 1114/1114 needed


In [6]:
try:
    from scipy import signal  # type: ignore

    SCIPY_AVAILABLE = True
except Exception as e:
    SCIPY_AVAILABLE = False
    signal = None
    print(
        "scipy not available; EEG filtering will fall back to no-filter path. Reason:",
        repr(e),
    )

if PLATFORM == "local":
    PATH_EEG = "./input/hms-harmful-brain-activity-classification/test_eegs/"
elif PLATFORM == "kaggle":
    PATH_EEG = "/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/"

files_eeg = os.listdir(PATH_EEG)
print(f"There are {len(files_eeg)} test eeg parquets")

eegs2 = {}
if SCIPY_AVAILABLE:
    b, a = signal.butter(3, np.float32([1, 40]) * 2 / SFREQ, "bandpass")

test_eeg_ids = set(test.eeg_id.astype(int).tolist())

for i, f in enumerate(files_eeg):
    if i % 100 == 0:
        print(i, ", ", end="")
    try:
        name = int(f.split(".")[0])
    except Exception:
        continue
    if name not in test_eeg_ids:
        continue

    raw_eeg = pd.read_parquet(f"{PATH_EEG}{f}")

    time_temp = 0
    time_start = round(time_temp + (50 - EEG_LENGTH) / 2 * 200)
    time_stop = round(time_temp + (50 + EEG_LENGTH) / 2 * 200)

    eeg_default = raw_eeg.loc[time_start : (time_stop - 1), :].reset_index(drop=True)

    list_eeg = []
    for region in BRAIN.keys():
        eeg = np.zeros((len(BRAIN[region]), eeg_default.shape[0]), dtype=np.float32)
        for chan_i, chan in enumerate(BRAIN[region]):
            eeg[chan_i, :] = (
                eeg_default.loc[:, chan.split("-")[0]]
                - eeg_default.loc[:, chan.split("-")[1]]
            ).values

        eeg[np.isnan(eeg)] = 0

        if SCIPY_AVAILABLE:
            if 200 != SFREQ:
                eeg = signal.resample_poly(eeg, SFREQ, 200, axis=1)
            eeg = signal.filtfilt(b, a, eeg, axis=1)

        eeg = (eeg - np.mean(eeg, 1, keepdims=True)) / (
            np.std(eeg, 1, keepdims=True) + 1e-6
        )
        list_eeg.append(np.reshape(eeg, (eeg.shape[0], eeg.shape[1], 1)))

    list_eeg = np.concatenate(list_eeg, 2)
    eegs2[name] = list_eeg

print("\nLoaded EEGs:", len(eegs2), "Loaded specs:", len(spectrograms2))



There are 1693 test eeg parquets
0 , 100 , 200 , 300 , 400 , 500 , 600 , 700 , 800 , 900 , 1000 , 1100 , 1200 , 1300 , 1400 , 1500 , 1600 , 
Loaded EEGs: 1693 Loaded specs: 1114


In [7]:
SUB_COLS = ["eeg_id"] + list(TARGETS)

preds = []
found_any = False

if not os.path.isdir(LOAD_MODELS_FROM):
    print(f"Model directory not found: {LOAD_MODELS_FROM}")
    weights_dir_exists = False
else:
    weights_dir_exists = True

if TF_AVAILABLE and weights_dir_exists:
    test_gen = DataGenerator(
        test, shuffle=False, batch_size=32, mode="test", specs=spectrograms2, eegs=eegs2
    )

    with strategy.scope():
        model = build_model()

    for i in range(5):
        wpath = os.path.join(LOAD_MODELS_FROM, f"EB2_v{VER}_f{i}.h5")
        if not os.path.exists(wpath):
            print(f"Fold {i+1}: weights not found at {wpath} (skipping)")
            continue

        print(f"Fold {i+1}: loading {wpath}")
        model.load_weights(wpath)
        pred_i = model.predict(test_gen, verbose=1)
        preds.append(pred_i)
        found_any = True

if found_any:
    pred = np.mean(preds, axis=0)
    print("\nTest preds shape", pred.shape)
    pred = np.nan_to_num(
        pred,
        nan=1.0 / len(TARGETS),
        posinf=1.0 / len(TARGETS),
        neginf=1.0 / len(TARGETS),
    )
    pred = np.clip(pred, 1e-8, 1.0)
    pred = pred / pred.sum(axis=1, keepdims=True)
else:
    print(
        "\nNo usable model predictions (either TF unavailable or weights missing). "
        "Using patient/global prior fallback to create a valid submission."
    )

    # --- Small score-oriented calibration (minimal change):
    # Blend patient prior with global prior to reduce overconfident patient-specific spikes,
    # which generally reduces KL on this task compared to a hard patient prior.
    blend = 0.15  # keep small to preserve semantics; only affects fallback path
    pred = np.zeros((len(test), len(TARGETS)), dtype="float64")
    for idx, pid in enumerate(test["patient_id"].values):
        if pid in _patient_prior.index:
            p = _patient_prior.loc[pid, list(TARGETS)].values.astype("float64")
            pred[idx] = (1.0 - blend) * p + blend * _global_prior
        else:
            pred[idx] = _global_prior

    pred = np.nan_to_num(
        pred,
        nan=1.0 / len(TARGETS),
        posinf=1.0 / len(TARGETS),
        neginf=1.0 / len(TARGETS),
    )
    pred = np.clip(pred, 1e-8, 1.0)
    pred = pred / pred.sum(axis=1, keepdims=True)

sub = pd.DataFrame({"eeg_id": test.eeg_id.values})
sub = sub.assign(**{c: pred[:, k].astype("float32") for k, c in enumerate(TARGETS)})
sub = sub[SUB_COLS]

# Final strict normalization/clipping to guarantee valid probabilities for submission checks
vals = sub[list(TARGETS)].values.astype("float64")
vals = np.nan_to_num(
    vals, nan=1.0 / len(TARGETS), posinf=1.0 / len(TARGETS), neginf=1.0 / len(TARGETS)
)
vals = np.clip(vals, 1e-8, 1.0)
vals = vals / vals.sum(axis=1, keepdims=True)
sub.loc[:, list(TARGETS)] = vals.astype("float32")

if PLATFORM == "local":
    sample_path = (
        "./input/hms-harmful-brain-activity-classification/sample_submission.csv"
    )
else:
    sample_path = (
        "/kaggle/input/hms-harmful-brain-activity-classification/sample_submission.csv"
    )
sample_cols = list(pd.read_csv(sample_path, nrows=1).columns)

sub = sub[sample_cols]
sub.to_csv("submission.csv", index=False)

print("Submission shape", sub.shape)
print("Row-sum stats:", sub[TARGETS].sum(axis=1).min(), sub[TARGETS].sum(axis=1).max())
print(sub.head())

Model directory not found: /kaggle/input/models20240117

No usable model predictions (either TF unavailable or weights missing). Using patient/global prior fallback to create a valid submission.
Submission shape (9850, 7)
Row-sum stats: 0.9999998807907104 1.0000001192092896
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  2578018731      0.028063  0.030749  0.612926   0.015181   0.056841   
1  2578018731      0.028063  0.030749  0.612926   0.015181   0.056841   
2  2578018731      0.028063  0.030749  0.612926   0.015181   0.056841   
3  2578018731      0.028063  0.030749  0.612926   0.015181   0.056841   
4  1706196508      0.100481  0.145565  0.027877   0.095172   0.040167   

   other_vote  
0    0.256240  
1    0.256240  
2    0.256240  
3    0.256240  
4    0.590738  
